# Hands-on Sesi 6

Nama Lengkap : Dikki Frana Alvian

NIM          : 240401010151

Kelas        : IF403

Link: https://colab.research.google.com/drive/1h_3o43HBNUEaRk75k7jFuZFHnBEhAoDh?usp=sharing

# 1. Load & EDA Singkat

In [1]:
import pandas as pd, seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# MUAT DATASET
df = sns.load_dataset('titanic')

# Pilih kolom yang akan digunakan
cols = ['pclass','sex','age','sibsp','parch','fare','embarked','survived']
df = df[cols].copy()

In [3]:
# PERIKSA MISSING VALUES
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())

Shape: (891, 8)

Missing values:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64


In [4]:
# TIPE DATA MASING-MASING KOLOM
print('Tipe data tiap kolom : \n')
print(df.info())

Tipe data tiap kolom : 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    891 non-null    int64  
 1   sex       891 non-null    object 
 2   age       714 non-null    float64
 3   sibsp     891 non-null    int64  
 4   parch     891 non-null    int64  
 5   fare      891 non-null    float64
 6   embarked  889 non-null    object 
 7   survived  891 non-null    int64  
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB
None


In [5]:
# PERIKSA DISTRIBUSI TARGET
print('\nDistribusi target:')
print(df['survived'].value_counts(normalize=True).round(3))


Distribusi target:
survived
0    0.616
1    0.384
Name: proportion, dtype: float64


In [6]:
# survived=0: ~61.6%, survived=1: ~38.4% — kelas tidak seimbang!

# 2. Handling Missing Values

In [7]:
# Age: isi dengan median (robust terhadap outlier)
df['age'] = df['age'].fillna(df['age'].median())

# Embarked: isi dengan modus (nilai paling sering)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

print('Missing setelah handling:')
print(df.isnull().sum()) # Semua harus 0

Missing setelah handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64


# 3. Encoding Kategorikal

In [8]:
# One-Hot Encoding untuk 'sex' dan 'embarked'
df = pd.get_dummies(df,
    columns=['sex', 'embarked'],
    drop_first=True, # hindari dummy variable trap
    dtype=int)

print('Kolom setelah encoding:')
print(df.columns.tolist())

# ['pclass','age','sibsp','parch','fare','survived', 'sex_male','embarked_Q','embarked_S']

Kolom setelah encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']


# 4. Train-Test Split

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y) # proporsi kelas terjaga

print(f'Train: {X_train.shape[0]} baris')
print(f'Test : {X_test.shape[0]} baris')
print('\nProporsi survived di Train:')
print(y_train.value_counts(normalize=True).round(3))
print('\nProporsi survived di Test:')
print(y_test.value_counts(normalize=True).round(3))

Train: 712 baris
Test : 179 baris

Proporsi survived di Train:
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Proporsi survived di Test:
survived
0    0.615
1    0.385
Name: proportion, dtype: float64


# 5. Feature Scaling

In [10]:
from sklearn.preprocessing import StandardScaler
# Hanya kolom numerik yang perlu di-scale
# Kolom biner (sex_male, embarked_Q, embarked_S) TIDAK perlu
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']

scaler = StandardScaler()

# fit_transform pada training set (belajar μ dan σ dari sini)
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# transform saja pada test set (gunakan μ dan σ dari training!)
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('Mean scaler (dari train):', scaler.mean_.round(2))
print('Std scaler (dari train):', scaler.scale_.round(2))
print()
print('Contoh X_train setelah scaling:')
print(X_train.head(3).round(3))

print('\nData siap dilatih model Machine Learning!')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}, y_test : {y_test.shape}')

Mean scaler (dari train): [ 2.31 29.46  0.49  0.39 31.82]
Std scaler (dari train): [ 0.83 13.03  1.06  0.84 48.03]

Contoh X_train setelah scaling:
     pclass    age  sibsp  parch   fare  sex_male  embarked_Q  embarked_S
692   0.830 -0.112 -0.465 -0.466  0.514         1           0           1
481  -0.371 -0.112 -0.465 -0.466 -0.663         1           0           1
527  -1.571 -0.112 -0.465 -0.466  3.955         1           0           1

Data siap dilatih model Machine Learning!
X_train: (712, 8), y_train: (712,)
X_test : (179, 8), y_test : (179,)


# Kesimpulan Sesi Pembelajaran Sesi 6

*   Apa yang telah dipelajari: Saya telah mempraktikkan tahap kritis Data Preparation dalam siklus Data Science. Materi yang dipelajari meliputi teknik Encoding (Label, One-Hot, Ordinal), Feature Scaling (StandardScaler), serta pentingnya melakukan Train-Test Split dengan stratifikasi sebelum melatih model Machine Learning.

*   Temuan utama: Pada dataset Titanic, ditemukan bahwa data mentah memiliki banyak nilai yang hilang pada kolom age. Setelah dilakukan preprocessing, data yang tadinya berupa teks (sex dan embarked) berhasil diubah menjadi format numerik melalui One-Hot Encoding. Penggunaan StandardScaler memastikan fitur-fitur seperti fare dan age memiliki skala yang seragam sehingga tidak mendominasi model nantinya.

*   Keterbatasan atau pertanyaan: Saya perlu memperdalam dengan adanya berbagai macam metode scaling, kapan waktu yang tepat untuk menggunakannya.